# H&M Customer Segmentation — Phase 3: Exploratory Data Analysis (EDA)

## Mục tiêu
Phase 3 chưa tạo model và chưa chọn feature cuối. Notebook này dùng dữ liệu sạch từ Phase 2 để trả lời:

1. Hoạt động mua hàng thay đổi theo thời gian thế nào?
2. Customer activity và spending phân bố ra sao?
3. Repeat purchase có đủ để tạo Frequency, Interpurchase Time và Purchase Regularity không?
4. Fashion/product metadata nào có variability và coverage tốt?
5. Sales channel behavior có đáng thử làm candidate feature không?
6. Candidate feature nào được EDA hỗ trợ để chuyển sang Phase 4?

> EDA chỉ quan sát và hình thành giả thuyết. Feature cuối sẽ được quyết định ở Feature Selection + Clustering Experiments.

## 0. Input / Output

Input:
```text
data/processed/
├── transactions_clean.parquet
├── customers_clean.parquet
└── articles_clean.parquet
```

Output:
```text
reports/
├── figures/phase3_eda/
└── tables/phase3_eda/
```

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import duckdb

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path("..")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TX_PATH = PROCESSED_DIR / "transactions_clean.parquet"
CUSTOMERS_PATH = PROCESSED_DIR / "customers_clean.parquet"
ARTICLES_PATH = PROCESSED_DIR / "articles_clean.parquet"

FIG_DIR = PROJECT_ROOT / "reports" / "figures" / "phase3_eda"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables" / "phase3_eda"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

for p in [TX_PATH, CUSTOMERS_PATH, ARTICLES_PATH]:
    print(f"{p}: {'OK' if p.exists() else 'MISSING'}")

con = duckdb.connect()
con.execute("SET max_memory = '3.5GB'")
con.execute("SET temp_directory = '../.duckdb_temp'")
con.execute("SET preserve_insertion_order = false")
customers = pd.read_parquet(CUSTOMERS_PATH)
articles = pd.read_parquet(ARTICLES_PATH)
tx_sql = f"SELECT * FROM read_parquet('{TX_PATH.as_posix()}')"
print("Phase 3 DuckDB initialized with safe 2.5GB memory limit.")

..\data\processed\transactions_clean.parquet: OK
..\data\processed\customers_clean.parquet: OK
..\data\processed\articles_clean.parquet: OK


Phase 3 DuckDB initialized with safe 2.5GB memory limit.


## 1. Dataset overview

In [2]:
overview = con.execute(f'''
SELECT
    COUNT(*) AS transaction_rows,
    COUNT(DISTINCT customer_id) AS customers,
    COUNT(DISTINCT article_id) AS articles,
    MIN(t_dat) AS min_date,
    MAX(t_dat) AS max_date,
    SUM(price) AS total_observed_spend,
    AVG(price) AS avg_item_price
FROM ({tx_sql})
''').df()

display(overview)
overview.to_csv(TABLE_DIR / "01_dataset_overview.csv", index=False)

,transaction_rows,customers,articles,min_date,max_date,total_observed_spend,avg_item_price
0,31788324,1362281,104547,2018-09-20,2020-09-22,884645.974054,0.027829


# A. Time-based EDA

## 2. Monthly activity
Theo dõi transaction rows, active customers, active articles và observed spend để phát hiện seasonality, partial months và biến động bất thường.

In [3]:
monthly = con.execute(f'''
SELECT
    DATE_TRUNC('month', t_dat) AS month,
    COUNT(*) AS transaction_rows,
    COUNT(DISTINCT customer_id) AS active_customers,
    COUNT(DISTINCT article_id) AS active_articles,
    SUM(price) AS observed_spend,
    AVG(price) AS avg_item_price
FROM ({tx_sql})
GROUP BY 1
ORDER BY 1
''').df()

monthly["month"] = pd.to_datetime(monthly["month"])
display(monthly)
monthly.to_csv(TABLE_DIR / "02_monthly_activity.csv", index=False)

for y, title, fname in [
    ("transaction_rows", "Monthly purchase activity", "01_monthly_transaction_rows.png"),
    ("active_customers", "Monthly active customers", "02_monthly_active_customers.png"),
    ("observed_spend", "Monthly observed spend", "03_monthly_observed_spend.png"),
]:
    fig, ax = plt.subplots(figsize=(11,5))
    ax.plot(monthly["month"], monthly[y], marker="o")
    ax.set_xlabel("Month")
    ax.set_ylabel(y.replace("_"," ").title())
    ax.set_title(title)
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(FIG_DIR / fname, dpi=160, bbox_inches="tight")
    plt.show()

,month,transaction_rows,active_customers,active_articles,observed_spend,avg_item_price
0,2018-09-01,594776,140340,25698,17755.639593,0.029853
1,2018-10-01,1397040,274220,33376,41584.816610,0.029766
2,2018-11-01,1270619,274207,31807,39000.988390,0.030694
3,2018-12-01,1148827,254059,32618,32532.514712,0.028318
4,2019-01-01,1263471,242024,32309,33449.981475,0.026475
5,2019-02-01,1152412,238992,32270,31428.221814,0.027272
6,2019-03-01,1286750,244166,31749,37549.020169,0.029181
7,2019-04-01,1476454,260875,33136,42824.987305,0.029005
8,2019-05-01,1560319,271554,34449,43357.502322,0.027788
9,2019-06-01,1906202,305190,33346,48648.417864,0.025521


C:\Users\Dell\AppData\Local\Temp\ipykernel_18632\1303407200.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Day-of-week behavior

In [4]:
dow = con.execute(f'''
SELECT
    EXTRACT('dow' FROM t_dat) AS dow,
    COUNT(*) AS transaction_rows,
    COUNT(DISTINCT customer_id) AS active_customers,
    SUM(price) AS observed_spend
FROM ({tx_sql})
GROUP BY 1
ORDER BY 1
''').df()

dow_map = {0:"Sunday",1:"Monday",2:"Tuesday",3:"Wednesday",4:"Thursday",5:"Friday",6:"Saturday"}
dow["day"] = dow["dow"].map(dow_map)
display(dow)
dow.to_csv(TABLE_DIR / "03_day_of_week_activity.csv", index=False)

fig, ax = plt.subplots(figsize=(9,5))
ax.bar(dow["day"], dow["transaction_rows"])
ax.set_xlabel("Day of week")
ax.set_ylabel("Transaction rows")
ax.set_title("Purchase activity by day of week")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(FIG_DIR / "04_day_of_week_activity.png", dpi=160, bbox_inches="tight")
plt.show()

,dow,transaction_rows,active_customers,observed_spend,day
0,0,4033592,536447,114570.167949,Sunday
1,1,4054054,569309,114883.797983,Monday
2,2,4198630,581964,119471.791661,Tuesday
3,3,4842821,612850,133354.508390,Wednesday
4,4,4860595,628388,135167.938305,Thursday
5,5,4749132,626868,130138.643254,Friday
6,6,5049500,627453,137059.126508,Saturday


C:\Users\Dell\AppData\Local\Temp\ipykernel_18632\3369361017.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# B. Customer activity EDA

## 4. Customer-level EDA table
Đây chỉ là bảng EDA, chưa phải modelling table.

In [5]:
customer_base = con.execute(f'''
SELECT
    customer_id,
    COUNT(*) AS item_rows,
    COUNT(DISTINCT t_dat) AS purchase_days,
    SUM(price) AS total_spend,
    AVG(price) AS avg_item_price,
    MIN(t_dat) AS first_purchase_date,
    MAX(t_dat) AS last_purchase_date,
    COUNT(DISTINCT article_id) AS distinct_articles
FROM ({tx_sql})
GROUP BY customer_id
''').df()

display(customer_base.head())
customer_base.to_parquet(TABLE_DIR / "04_customer_base_eda.parquet", index=False)

customer_summary = customer_base[
    ["item_rows","purchase_days","total_spend","avg_item_price","distinct_articles"]
].describe(percentiles=[.01,.05,.25,.5,.75,.9,.95,.99]).T
display(customer_summary)
customer_summary.to_csv(TABLE_DIR / "05_customer_activity_summary.csv")

,customer_id,item_rows,purchase_days,total_spend,avg_item_price,first_purchase_date,last_purchase_date,distinct_articles
0,c7217359fdd4733d518db61dc8e03b2d348a656358c274...,3,3,0.123678,0.041226,2018-10-08,2019-05-20,3
1,c740266013e5d47b8e581463325ee3e47e69b715532f68...,45,22,0.844271,0.018762,2018-09-29,2020-09-22,45
2,c7b5900bd2de5d46a908f281eb934109727d683d8ee920...,159,31,4.808525,0.030242,2018-09-29,2020-08-23,137
3,c8d866bb9f5fbd03dc3aae6909f440887d9fd2835a2565...,4,1,0.115186,0.028797,2018-10-08,2018-10-08,4
4,c9e86f9384fab9115e8eed5495b6ec491d514328ad2b01...,37,15,0.871407,0.023552,2018-10-08,2020-08-23,36


,count,mean,std,min,1%,5%,25%,50%,75%,90%,95%,99%,max
item_rows,1362281.0,23.334631,39.242253,1.000000,1.000000,1.000000,3.000000,9.000000,27.000000,60.000000,91.000000,187.000000,1895.000000
purchase_days,1362281.0,6.665423,9.707005,1.000000,1.000000,1.000000,1.000000,3.000000,8.000000,17.000000,24.000000,45.000000,427.000000
total_spend,1362281.0,0.649386,1.197582,0.000763,0.013542,0.027424,0.088068,0.245593,0.701186,1.622898,2.573542,5.698885,57.676407
avg_item_price,1362281.0,0.028755,0.014293,0.000763,0.008458,0.013542,0.020958,0.026133,0.033361,0.042356,0.050831,0.084729,0.506780
distinct_articles,1362281.0,20.044645,31.980251,1.000000,1.000000,1.000000,3.000000,8.000000,24.000000,52.000000,77.000000,153.000000,1346.000000


## 5. Repeat-purchase structure

In [6]:
thresholds = [1,2,3,5,10]
repeat_summary = pd.DataFrame({
    "threshold_purchase_days": thresholds,
    "customers": [int((customer_base["purchase_days"] >= k).sum()) for k in thresholds],
    "pct_customers": [float((customer_base["purchase_days"] >= k).mean()*100) for k in thresholds]
})
display(repeat_summary)
repeat_summary.to_csv(TABLE_DIR / "06_repeat_purchase_thresholds.csv", index=False)

plot_s = customer_base["purchase_days"].clip(upper=customer_base["purchase_days"].quantile(.99))
fig, ax = plt.subplots(figsize=(9,5))
ax.hist(plot_s, bins=50)
ax.set_xlabel("Distinct purchase days/customer (clipped at p99)")
ax.set_ylabel("Customers")
ax.set_title("Customer purchase-frequency distribution")
fig.tight_layout()
fig.savefig(FIG_DIR / "05_purchase_days_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

,threshold_purchase_days,customers,pct_customers
0,1,1362281,100.000000
1,2,914308,67.115962
2,3,731192,53.674095
3,5,534068,39.203953
4,10,294802,21.640322


C:\Users\Dell\AppData\Local\Temp\ipykernel_18632\393632308.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Spending distribution

In [7]:
spend = customer_base["total_spend"].dropna()

fig, ax = plt.subplots(figsize=(9,5))
ax.hist(spend.clip(upper=spend.quantile(.99)), bins=50)
ax.set_xlabel("Total spend/customer (clipped at p99)")
ax.set_ylabel("Customers")
ax.set_title("Customer total spend distribution")
fig.tight_layout()
fig.savefig(FIG_DIR / "06_customer_total_spend_p99.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9,5))
ax.hist(np.log1p(spend), bins=50)
ax.set_xlabel("log1p(total spend)")
ax.set_ylabel("Customers")
ax.set_title("Customer total spend on log scale")
fig.tight_layout()
fig.savefig(FIG_DIR / "07_customer_total_spend_log.png", dpi=160, bbox_inches="tight")
plt.show()

C:\Users\Dell\AppData\Local\Temp\ipykernel_18632\755210758.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Dell\AppData\Local\Temp\ipykernel_18632\755210758.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Spend concentration

In [8]:
sorted_spend = customer_base.sort_values("total_spend", ascending=False)
total = sorted_spend["total_spend"].sum()
rows = []
for pct in [0.01,0.05,0.10,0.20]:
    n = max(1, int(np.ceil(len(sorted_spend)*pct)))
    rows.append({
        "top_customer_pct": pct*100,
        "customer_count": n,
        "share_of_observed_spend_pct": sorted_spend.head(n)["total_spend"].sum()/total*100
    })
concentration = pd.DataFrame(rows)
display(concentration)
concentration.to_csv(TABLE_DIR / "07_spend_concentration.csv", index=False)

fig, ax = plt.subplots(figsize=(8,5))
ax.bar(concentration["top_customer_pct"].astype(str)+"%", concentration["share_of_observed_spend_pct"])
ax.set_xlabel("Top customer group")
ax.set_ylabel("Share of observed spend (%)")
ax.set_title("Observed spend concentration")
fig.tight_layout()
fig.savefig(FIG_DIR / "08_spend_concentration.png", dpi=160, bbox_inches="tight")
plt.show()

,top_customer_pct,customer_count,share_of_observed_spend_pct
0,1.0,13623,13.181178
1,5.0,68115,35.545345
2,10.0,136229,51.123394
3,20.0,272457,69.595215


C:\Users\Dell\AppData\Local\Temp\ipykernel_18632\2933276284.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# C. Temporal customer behavior

## 8. Interpurchase intervals
Chỉ customer có từ hai distinct purchase dates mới có interval. Ta kiểm tra coverage và distribution trước khi quyết định dùng `T`.

In [9]:
intervals = con.execute(f'''
WITH days AS (
    SELECT DISTINCT customer_id, t_dat
    FROM ({tx_sql})
),
ordered AS (
    SELECT
        customer_id,
        t_dat,
        LAG(t_dat) OVER (PARTITION BY customer_id ORDER BY t_dat) AS prev_date
    FROM days
),
diffs AS (
    SELECT
        customer_id,
        DATE_DIFF('day', prev_date, t_dat) AS interval_days
    FROM ordered
    WHERE prev_date IS NOT NULL
)
SELECT
    customer_id,
    COUNT(*) AS interval_count,
    AVG(interval_days) AS mean_interpurchase_days,
    MEDIAN(interval_days) AS median_interpurchase_days,
    STDDEV_SAMP(interval_days) AS std_interpurchase_days,
    MIN(interval_days) AS min_interpurchase_days,
    MAX(interval_days) AS max_interpurchase_days
FROM diffs
GROUP BY customer_id
''').df()

display(intervals.head())
intervals.to_parquet(TABLE_DIR / "08_interpurchase_interval_eda.parquet", index=False)

temporal_coverage = pd.DataFrame({
    "metric": [
        "total_customers",
        "customers_with_at_least_1_interval",
        "mean_interval_coverage_pct",
        "customers_with_std_interval",
        "regularity_coverage_pct"
    ],
    "value": [
        len(customer_base),
        len(intervals),
        len(intervals)/len(customer_base)*100,
        int(intervals["std_interpurchase_days"].notna().sum()),
        intervals["std_interpurchase_days"].notna().sum()/len(customer_base)*100
    ]
})
display(temporal_coverage)
temporal_coverage.to_csv(TABLE_DIR / "09_temporal_feature_coverage.csv", index=False)

s = intervals["mean_interpurchase_days"].dropna()
fig, ax = plt.subplots(figsize=(9,5))
ax.hist(s.clip(upper=s.quantile(.99)), bins=50)
ax.set_xlabel("Mean interpurchase interval (days, clipped at p99)")
ax.set_ylabel("Customers")
ax.set_title("Interpurchase-time distribution")
fig.tight_layout()
fig.savefig(FIG_DIR / "09_interpurchase_time_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

,customer_id,interval_count,mean_interpurchase_days,median_interpurchase_days,std_interpurchase_days,min_interpurchase_days,max_interpurchase_days
0,c8fdb67c9512fed52ee318e9a7ddcfc495730bf4030af4...,21,33.000000,21.0,31.866911,4,139
1,c8fe1a898a089fcb49564456f0bee490bcedab593d5639...,1,213.000000,213.0,NaN,213,213
2,c90566a84d28d904872b62a6c23500a54424759b3d3dd0...,2,65.000000,65.0,45.254834,33,97
3,c9074a9e9078b1ae63c730acafa9b73a0da048c95a466c...,1,211.000000,211.0,NaN,211,211
4,c908943407710ac5ac28c097238cae3b80fbbab4f779a9...,14,39.071429,29.5,62.372904,2,249


,metric,value
0,total_customers,1.362281e+06
1,customers_with_at_least_1_interval,9.143080e+05
2,mean_interval_coverage_pct,6.711596e+01
3,customers_with_std_interval,7.311920e+05
4,regularity_coverage_pct,5.367410e+01


C:\Users\Dell\AppData\Local\Temp\ipykernel_18632\2688568178.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# D. Fashion / product EDA

## 9. Catalog structure

In [10]:
fashion_dims = [
    c for c in [
        "product_type_name","product_group_name","department_name",
        "section_name","garment_group_name","colour_group_name"
    ] if c in articles.columns
]

catalog_structure = pd.DataFrame({
    "feature": fashion_dims,
    "n_unique": [articles[c].nunique(dropna=True) for c in fashion_dims],
    "missing_pct": [articles[c].isna().mean()*100 for c in fashion_dims]
}).sort_values("n_unique", ascending=False)

display(catalog_structure)
catalog_structure.to_csv(TABLE_DIR / "10_catalog_structure.csv", index=False)

,feature,n_unique,missing_pct
2,department_name,250,0.0
0,product_type_name,131,0.0
3,section_name,56,0.0
5,colour_group_name,50,0.0
4,garment_group_name,21,0.0
1,product_group_name,19,0.0


## 10. Top purchased fashion groups

In [11]:
fashion_code_cols = [c for c in ["product_type_no","garment_group_no","colour_group_code","department_no","section_no"] if c in articles.columns]
article_join_cols = list(dict.fromkeys(["article_id"] + fashion_dims + fashion_code_cols))
con.register("articles_dim_phase3", articles[article_join_cols])

def top_dimension(col, n=15):
    return con.execute(f'''
        SELECT
            COALESCE(a.{col}, 'UNKNOWN') AS category,
            COUNT(*) AS transaction_rows,
            COUNT(DISTINCT tx.customer_id) AS customers
        FROM ({tx_sql}) tx
        LEFT JOIN articles_dim_phase3 a ON tx.article_id = a.article_id
        GROUP BY 1
        ORDER BY transaction_rows DESC
        LIMIT {n}
    ''').df()

for col in ["product_type_name","garment_group_name","section_name","colour_group_name"]:
    if col in fashion_dims:
        df = top_dimension(col)
        display(df)
        df.to_csv(TABLE_DIR / f"11_top_{col}.csv", index=False)
        fig, ax = plt.subplots(figsize=(10,6))
        ax.barh(df["category"], df["transaction_rows"])
        ax.set_xlabel("Transaction rows")
        ax.set_title(f"Top purchased {col}")
        ax.invert_yaxis()
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"10_top_{col}.png", dpi=160, bbox_inches="tight")
        plt.show()

,category,transaction_rows,customers
0,Trousers,4217017,738844
1,Dress,3238428,608936
2,Sweater,2783274,642456
3,T-shirt,2203750,558353
4,Top,1583408,506862
5,Blouse,1504868,441263
6,Vest top,1414101,424945
7,Bra,1335233,389371
8,Shorts,1152513,358567
9,Bikini top,1126202,336755


C:\Users\Dell\AppData\Local\Temp\ipykernel_18632\2505560184.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,category,transaction_rows,customers
0,Jersey Fancy,5171611,834125
1,Jersey Basic,3268451,659032
2,"Under-, Nightwear",2966788,577482
3,Trousers,2953711,639221
4,Swimwear,2538085,422796
5,Blouses,2510441,581614
6,Knitwear,2392807,593284
7,Dresses Ladies,2147529,508023
8,Accessories,1645464,502218
9,Trousers Denim,1279886,389193


C:\Users\Dell\AppData\Local\Temp\ipykernel_18632\2505560184.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,category,transaction_rows,customers
0,Womens Everyday Collection,5860832,817652
1,Divided Collection,3768107,687432
2,"Womens Swimwear, beachwear",2455138,408567
3,Womens Lingerie,2200139,449645
4,Womens Tailoring,2133999,521848
5,Womens Everyday Basics,1541864,449875
6,Divided Basics,1290216,416841
7,Womens Casual,1228424,408381
8,Ladies H&M Sport,1177490,304358
9,"Womens Nightwear, Socks & Tigh",1110181,376833


C:\Users\Dell\AppData\Local\Temp\ipykernel_18632\2505560184.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,category,transaction_rows,customers
0,Black,11036956,1105112
1,White,3368276,744734
2,Dark Blue,2180620,619913
3,Light Beige,1263385,454764
4,Blue,1088455,424533
5,Beige,921184,376248
6,Light Blue,913496,388783
7,Light Pink,858234,375591
8,Off White,841673,370041
9,Grey,840065,391449


C:\Users\Dell\AppData\Local\Temp\ipykernel_18632\2505560184.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Customer fashion diversity

In [12]:
art_path = ARTICLES_PATH.as_posix()
tx_p = TX_PATH.as_posix()

# Direct scan with integer codes - zero temp table, zero disk spill
div1 = con.execute(f'''
SELECT
    tx.customer_id,
    COUNT(DISTINCT tx.article_id) AS product_diversity,
    COUNT(DISTINCT a.garment_group_no) AS garment_group_diversity,
    COUNT(DISTINCT a.colour_group_code) AS colour_diversity
FROM read_parquet('{tx_p}') tx
JOIN read_parquet('{art_path}') a USING (article_id)
GROUP BY tx.customer_id
''').df()

div2 = con.execute(f'''
SELECT
    tx.customer_id,
    COUNT(DISTINCT a.product_type_no) AS product_type_diversity,
    COUNT(DISTINCT a.department_no) AS department_diversity
FROM read_parquet('{tx_p}') tx
JOIN read_parquet('{art_path}') a USING (article_id)
GROUP BY tx.customer_id
''').df()

cust_fashion = div1.merge(div2, on="customer_id")
print("Customer fashion diversity (Phase 3):", cust_fashion.shape)
display(cust_fashion.head())
fashion_diversity = cust_fashion

Customer fashion diversity (Phase 3): (1362281, 6)


,customer_id,product_diversity,garment_group_diversity,colour_diversity,product_type_diversity,department_diversity
0,ae0e0a22e5d72f5f4cb2e7e659f68d16c23d751e1f882f...,33,11,17,14,19
1,b7218cd929488aff1e8edd32831ba577e28f7759031aa0...,17,7,9,9,8
2,c6dcb346a5f620bf47b3af32889abfd675e0701496603e...,326,18,30,43,74
3,c95c80892ae72976ed3d595184875d76f60377bc9eafdc...,15,8,6,9,12
4,d5b5757606005417f24afe6731d0586c78b324241c5fce...,126,15,23,20,40


# E. Sales channel EDA

## 12. Channel behavior
Không tự gán nhãn business cho `sales_channel_id` nếu chưa xác minh mapping.

In [13]:
channel_global = con.execute(f'''
SELECT
    sales_channel_id,
    COUNT(*) AS transaction_rows,
    COUNT(DISTINCT customer_id) AS customers,
    SUM(price) AS observed_spend
FROM ({tx_sql})
GROUP BY 1
ORDER BY transaction_rows DESC
''').df()

display(channel_global)
channel_global.to_csv(TABLE_DIR / "14_channel_global.csv", index=False)

fig, ax = plt.subplots(figsize=(8,5))
ax.bar(channel_global["sales_channel_id"].astype(str), channel_global["transaction_rows"])
ax.set_xlabel("sales_channel_id")
ax.set_ylabel("Transaction rows")
ax.set_title("Purchase activity by sales channel ID")
fig.tight_layout()
fig.savefig(FIG_DIR / "12_sales_channel_distribution.png", dpi=160, bbox_inches="tight")
plt.show()

channel_customer = con.execute(f'''
WITH counts AS (
    SELECT customer_id, sales_channel_id, COUNT(*) AS n
    FROM ({tx_sql})
    GROUP BY customer_id, sales_channel_id
),
totals AS (
    SELECT customer_id, SUM(n) AS total_n, COUNT(*) AS n_channels, MAX(n) AS dominant_n
    FROM counts
    GROUP BY customer_id
)
SELECT customer_id, n_channels, dominant_n * 1.0 / total_n AS dominant_channel_share
FROM totals
''').df()

display(channel_customer.head())
channel_customer[["n_channels","dominant_channel_share"]].describe(
    percentiles=[.25,.5,.75,.9,.95,.99]
).T.to_csv(TABLE_DIR / "15_customer_channel_summary.csv")

,sales_channel_id,transaction_rows,customers,observed_spend
0,2,22379862,1108418,668963.146136
1,1,9408462,737118,215682.827915


C:\Users\Dell\AppData\Local\Temp\ipykernel_18632\1947597397.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,customer_id,n_channels,dominant_channel_share
0,2e1b8a8efd7b21b17fbde697d62057d23726fb0e7be779...,1,1.000000
1,66ced930608d3ba937b48146445ff3bee25b79731e2fc6...,2,0.687500
2,7d7ec1c1c97591029ec16b3e1811350778e35280a126ff...,1,1.000000
3,aa75b60209bbe3450dda56868856e8ace961004e2506af...,2,0.603448
4,b54fc7d0cb66bcbe7a89950c9350350110c397e31552aa...,2,0.643172


# F. Customer metadata EDA

## 13. Age and membership/news variables
Demographic metadata có thể chỉ dùng để profiling nếu không phù hợp làm clustering feature.

In [14]:
if "age" in customers.columns:
    age = pd.to_numeric(customers["age"], errors="coerce").dropna()
    fig, ax = plt.subplots(figsize=(9,5))
    ax.hist(age, bins=40)
    ax.set_xlabel("Age")
    ax.set_ylabel("Customers")
    ax.set_title("Customer age distribution")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "13_age_distribution.png", dpi=160, bbox_inches="tight")
    plt.show()

for c in ["club_member_status","fashion_news_frequency","FN","Active"]:
    if c in customers.columns:
        vc = customers[c].value_counts(dropna=False).rename_axis(c).reset_index(name="count")
        display(vc)
        vc.to_csv(TABLE_DIR / f"16_{c}_distribution.csv", index=False)

C:\Users\Dell\AppData\Local\Temp\ipykernel_18632\2604832787.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,club_member_status,count
0,ACTIVE,1272491
1,PRE-CREATE,92960
2,<NA>,6062
3,LEFT CLUB,467


,fashion_news_frequency,count
0,NONE,877711
1,REGULARLY,477416
2,<NA>,16011
3,MONTHLY,842


,FN,count
0,NaN,895050
1,1.0,476930


,Active,count
0,NaN,907576
1,1.0,464404


# G. Relationships between customer behaviors

## 14. Spearman correlation diagnostic
Đây chưa phải Feature Selection chính thức; chỉ phát hiện quan hệ mạnh/redundancy tiềm năng.

In [15]:
eda_customer = (
    customer_base
    .merge(fashion_diversity, on="customer_id", how="left")
    .merge(intervals, on="customer_id", how="left")
    .merge(channel_customer, on="customer_id", how="left")
)

numeric_cols = eda_customer.select_dtypes(include=[np.number]).columns.tolist()
corr = eda_customer[numeric_cols].corr(method="spearman")
display(corr.round(3))
corr.to_csv(TABLE_DIR / "17_customer_spearman_correlation.csv")

fig, ax = plt.subplots(figsize=(12,10))
im = ax.imshow(corr.values, aspect="auto")
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)
ax.set_title("Spearman correlation — EDA customer variables")
fig.colorbar(im, ax=ax, label="Spearman correlation")
fig.tight_layout()
fig.savefig(FIG_DIR / "14_customer_spearman_correlation.png", dpi=160, bbox_inches="tight")
plt.show()

,item_rows,purchase_days,total_spend,avg_item_price,distinct_articles,product_diversity,garment_group_diversity,colour_diversity,product_type_diversity,department_diversity,interval_count,mean_interpurchase_days,median_interpurchase_days,std_interpurchase_days,min_interpurchase_days,max_interpurchase_days,n_channels,dominant_channel_share
item_rows,1.000,0.889,0.959,-0.053,0.988,0.988,0.931,0.949,0.944,0.953,0.862,-0.366,-0.401,-0.389,-0.606,-0.024,0.591,-0.545
purchase_days,0.889,1.000,0.871,0.004,0.890,0.890,0.878,0.860,0.881,0.887,1.000,-0.425,-0.476,-0.447,-0.715,-0.016,0.665,-0.627
total_spend,0.959,0.871,1.000,0.205,0.947,0.947,0.903,0.910,0.904,0.920,0.831,-0.350,-0.384,-0.367,-0.587,-0.019,0.572,-0.526
avg_item_price,-0.053,0.004,0.205,1.000,-0.057,-0.057,-0.018,-0.052,-0.063,-0.030,0.005,-0.002,-0.003,-0.005,-0.013,-0.001,-0.031,0.039
distinct_articles,0.988,0.890,0.947,-0.057,1.000,1.000,0.946,0.963,0.960,0.968,0.869,-0.356,-0.386,-0.395,-0.591,-0.018,0.599,-0.554
product_diversity,0.988,0.890,0.947,-0.057,1.000,1.000,0.946,0.963,0.960,0.968,0.869,-0.356,-0.386,-0.395,-0.591,-0.018,0.599,-0.554
garment_group_diversity,0.931,0.878,0.903,-0.018,0.946,0.946,1.000,0.924,0.964,0.980,0.845,-0.335,-0.360,-0.378,-0.562,-0.012,0.606,-0.566
colour_diversity,0.949,0.860,0.910,-0.052,0.963,0.963,0.924,1.000,0.927,0.943,0.826,-0.342,-0.369,-0.375,-0.561,-0.022,0.582,-0.539
product_type_diversity,0.944,0.881,0.904,-0.063,0.960,0.960,0.964,0.927,1.000,0.968,0.851,-0.337,-0.362,-0.384,-0.564,-0.013,0.611,-0.570
department_diversity,0.953,0.887,0.920,-0.030,0.968,0.968,0.980,0.943,0.968,1.000,0.863,-0.344,-0.370,-0.391,-0.576,-0.013,0.606,-0.564


C:\Users\Dell\AppData\Local\Temp\ipykernel_18632\3924629399.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# H. Candidate Feature Evidence Table

## 15. Tổng hợp bằng chứng từ EDA
Bảng này chưa chọn final feature. Nó chỉ quyết định feature nào đủ điều kiện đi tiếp sang Phase 4.

In [16]:
n_customers = len(customer_base)
rows = [
    ["Recency","RFM",100.0,"How recently the customer purchased","PROCEED"],
    ["Frequency_purchase_days","RFM",100.0,"Distinct purchase dates","PROCEED"],
    ["Monetary_total_spend","RFM",100.0,"Observed cumulative spending","PROCEED"],
    ["MeanInterpurchaseTime","Temporal",len(intervals)/n_customers*100,"Typical time between purchase dates","PROCEED_WITH_CAUTION"],
    ["PurchaseRegularity","Temporal",intervals["std_interpurchase_days"].notna().sum()/n_customers*100,"Consistency of purchase timing","PROCEED_WITH_CAUTION"],
    ["AvgItemPrice","Spending",customer_base["avg_item_price"].notna().mean()*100,"Average purchased item price","PROCEED"],
    ["ProductDiversity","Fashion",fashion_diversity["product_diversity"].notna().mean()*100,"Distinct articles purchased","PROCEED"],
    ["DominantChannelShare","Channel",channel_customer["dominant_channel_share"].notna().mean()*100,"Strength of channel preference","PROCEED_WITH_CAUTION"]
]

for c,label in [
    ("product_type_diversity","ProductTypeDiversity"),
    ("garment_group_diversity","GarmentGroupDiversity"),
    ("colour_diversity","ColourDiversity"),
    ("department_diversity","DepartmentDiversity")
]:
    if c in fashion_diversity.columns:
        rows.append([label,"Fashion",fashion_diversity[c].notna().mean()*100,f"Customer diversity measured by {c}","PROCEED_WITH_CAUTION"])

candidate_evidence = pd.DataFrame(rows, columns=[
    "feature","group","coverage_pct","business_meaning","eda_decision"
])
display(candidate_evidence)
candidate_evidence.to_csv(TABLE_DIR / "18_candidate_feature_eda_evidence.csv", index=False)

,feature,group,coverage_pct,business_meaning,eda_decision
0,Recency,RFM,100.000000,How recently the customer purchased,PROCEED
1,Frequency_purchase_days,RFM,100.000000,Distinct purchase dates,PROCEED
2,Monetary_total_spend,RFM,100.000000,Observed cumulative spending,PROCEED
3,MeanInterpurchaseTime,Temporal,67.115962,Typical time between purchase dates,PROCEED_WITH_CAUTION
4,PurchaseRegularity,Temporal,53.674095,Consistency of purchase timing,PROCEED_WITH_CAUTION
5,AvgItemPrice,Spending,100.000000,Average purchased item price,PROCEED
6,ProductDiversity,Fashion,100.000000,Distinct articles purchased,PROCEED
7,DominantChannelShare,Channel,100.000000,Strength of channel preference,PROCEED_WITH_CAUTION
8,ProductTypeDiversity,Fashion,100.000000,Customer diversity measured by product_type_di...,PROCEED_WITH_CAUTION
9,GarmentGroupDiversity,Fashion,100.000000,Customer diversity measured by garment_group_d...,PROCEED_WITH_CAUTION


# I. Report-ready findings template

Sau khi chạy notebook, điền số liệu thật:

> Phân tích khám phá cho thấy hành vi mua hàng và spending của khách hàng có phân bố lệch đáng kể. Điều này là cơ sở để Phase Transformation kiểm tra log transformation/robust scaling thay vì loại extreme customers.
>
> ...% khách hàng có ít nhất hai purchase days và ...% có ít nhất ba purchase days. `Mean Interpurchase Time` có coverage ...%, còn `Purchase Regularity` có coverage ...%. Kết quả này quyết định liệu temporal features có được đưa sang candidate pool hay chỉ dùng cho subgroup.
>
> Các fashion-diversity variables ... có coverage và variability phù hợp để tiếp tục thử nghiệm, nhưng chưa được chọn làm final features.
>
> Sales-channel behavior cho thấy ..., vì vậy `DominantChannelShare` [được/không được] giữ làm candidate feature.

# J. Điều kiện chuyển sang Phase 4 — RFM Baseline + Extended Feature Engineering

- [ ] Monthly activity đã kiểm tra.
- [ ] Partial first/last month đã ghi nhận nếu có.
- [ ] Repeat-purchase rate đã biết.
- [ ] Coverage của Interpurchase Time đã biết.
- [ ] Coverage của Purchase Regularity đã biết.
- [ ] Spending skew đã mô tả.
- [ ] Fashion diversity distributions đã kiểm tra.
- [ ] Channel behavior đã kiểm tra.
- [ ] Candidate Feature Evidence Table đã sinh.
- [ ] Không chọn feature cuối chỉ từ EDA.

## Phase 4
1. Chốt công thức R, F, M.
2. Chọn snapshot/reference date.
3. Xây RFM baseline customer table.
4. Tạo candidate Extended features.
5. Kiểm tra coverage/missing sau feature engineering.
6. Tạo master customer feature table.
7. Xuất bảng và biểu đồ cho báo cáo.